In [42]:
def print_translations():
    with open("./en-zh.en-filtered.en.subword.test.desubword", 'r', encoding='utf-8') as en_file, \
         open("./zh.translated.desubword", 'r', encoding='utf-8') as zh_salient_file, \
         open("./zh.base.translated.desubword", 'r', encoding='utf-8') as zh_base_file, \
         open("./en-zh.zh-filtered.zh.subword.test.desubword", encoding="utf-8") as zh_correct_file:

        for en_line, zh_salient_line, zh_base_line, zh_correct_line in zip(en_file, zh_salient_file, zh_base_file, zh_correct_file):
            en_line = en_line.strip()
            zh_salient_line = zh_salient_line.strip()
            zh_base_line = zh_base_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            print(f"English:            {en_line}")
            print(f"Chinese (salient):  {zh_salient_line}")
            print(f"Chinese (base):     {zh_base_line}")
            print(f"Chinese (correct):  {zh_correct_line}")
            print("-" * 50)

# print_translations()

## Base model (post-finetuning) Analysis

In [6]:
from analysis_helper import *

en_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_trans_path = "./zh.base.translated.desubword"
zh_correct_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

df_sentences, df_mismatches = analyze_translations(en_path, zh_trans_path, zh_correct_path)

### Sort by METEOR (asc), edit distance (desc)

Edit distance is used as well to highlight sentences that are semantically different

In [40]:
df_sentences.sort_values(by=['METEOR', 'EditDistance'], ascending=[True, False]).head()

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
1497,RF: Here's it going down a pathway.,RF:现在沿着这条道路往下走。,"啊,我喜欢它--鲍勃,他正在下坡",0.0,0.0,16
555,And Laurie's going to talk about this one a li...,Laurie将会有一点点介绍。,劳里待会儿会说点这方面的东西,0.0,0.0,15
824,It's not civically rich enough for them to go ...,公民教育无法充分利用这些资源。,"那儿太冷清了,他们没兴趣",0.0,0.0,15
487,"Now, this is a conversation that often calls u...",这样的对话经常 带来许多罪恶感,这个话题时常让我们觉得很内疚。,0.0,0.0,14
281,Thank you. Namaste.,谢谢。纳姆斯特。,"而平凡如你我,只有两只眼睛",0.0,0.0,13


### Sort sentences based on input length

In [41]:
df_sentences.sort_values(by='English', key=lambda x: x.str.len()).head()

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
1316,36!,36!,你知道有多少种吗?36!,0.225904,0.009549,9
853,Hope.,希望,那就是希望,0.178571,0.024066,3
1925,Five.,五,观众:5。阿瑟:5。,0.000000,0.000000,10
230,Awful.,糟糕。,糟糕透了。,0.263158,0.065419,2
525,Enough.,够了。,够了。,0.981481,0.562341,0


### Result findings
- Mistranslated Entity
- Not capturing the correct sense
  - e.g And in summer, here, killer wasps.
  - e.g 法律人员 vs 合法的人 in "Humans and legal persons are not synonymous."
- Cannot translate well if 1 token for input (?)
  - e.g Us.;美;	是我们自己。
- Captured sentences literally (without considering enough context) 
  - Not good because the model is suppose to look at the entire corpus (?)
  - e.g It was all about going for the center.
- Wrong inversion
  - e.g Nobody wants to buy a mini well when they buy a car.
- Inaccuracy in capturing the correct sense
  - e.g It's a mind setting.
- Wrong intensity of adjective
  - e.g I was less exotic in this Whitopia.

## Base (fine-tuned) vs Salient Model Analysis

In [50]:
en_path = "./en-zh.en-filtered-salient.en.subword.test.desubword"
zh_trans_path = "./zh.translated.desubword"
zh_correct_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

df_salient_sentences, df_salient_mismatches = analyze_translations(en_path, zh_trans_path, zh_correct_path, salient=True)

In [59]:
# df_res = pd.merge(df_sentences, df_salient_sentences, on='Chinese (correct)', how='left', suffixes=('', '_salient'))
# df_res.to_csv("./analysis_results/combined_res.csv", index=False)
df_res = pd.read_csv("./analysis_results/combined_res.csv")
df_res.head()

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance,English_salient,Chinese (model)_salient,METEOR_salient,BLEU_salient,EditDistance_salient
0,"Dictatorships in Czechoslovakia, East Germany,...","在捷克洛伐克的达基地, 东德国,爱沙尼亚、拉塔维亚、马斯达加、马里斯加、马达加斯加、 宾夕法...","在捷克斯洛伐克,东德 爱沙尼亚,拉脱维亚,立陶宛, 马里,马达加斯加, 波兰,菲律宾, 塞尔...",0.177795,0.017233,51,believe don just hasn happened __SEP__ Dictato...,"在捷克斯洛伐克,东德国, 爱沙尼亚、拉特维亚、马里兰、马里亚、马达加斯加、 菲律宾、塞罗维亚...",0.246113,0.046779,40
1,I don't know what they're going to do with all...,我不知道他们要怎么处理那些东西。,我不知道他们将怎么处理那些东西。,0.895062,0.658037,1,thousands plastic trying people use __SEP__ I ...,我不知道他们要怎么处理这些东西。,0.778906,0.354948,2
2,It demands some sort of notion of inquiry beca...,它要求某种看证的概念 是一个没有雕刻的系统。,这需要有探求精神 因为这整个系统不是以雕塑形式做成的,0.094937,0.016296,23,logic building discussion idea started __SEP__...,"它需要一些审美的概念, 因为它是一个没有雕刻的系统。",0.231056,0.030588,23
3,"Great teachers do that, but what great teacher...","伟大的教师这样做,但是伟大的教师们也是这样, 指导、激励和投入。","优秀的教师的确要这样做 但同时他们还会指导学生的学习 激发学生的兴趣,挑起学生的热情,赢得学...",0.149502,0.021137,41,going education learning end teaching __SEP__ ...,"伟大的老师这么做了 但是伟大的老师也在做的 是辅导,激励着,激励着,激励着,激励着",0.129450,0.010157,41
4,And so I'm just plugging and chugging through ...,"所以我只是在方圆柱的任务上加力减速, 最终,在4000次努力中, 当我靠近我的正常线时, 我...",我按部就班地执行艰巨的任务 终于在第4000次筛选的时候 在我快发疯的时候 找到了符合标准的蛋白质,0.266472,0.021328,40,protein stages earliest levels cancer __SEP__ ...,"所以我只是插上插入这个树枝型的工作, 最后,在4000个尝试中, 当我接近失去理智的时候,我...",0.186441,0.017731,43


In [53]:
import pandas as pd
metrics = {
    "METEOR": df_res['METEOR'].mean(),
    "EditDistance": df_res['EditDistance'].mean(),
    "BLEU": df_res['BLEU'].mean(),
    "METEOR_salient": df_res['METEOR_salient'].mean(),
    "EditDistance_salient": df_res['EditDistance_salient'].mean(),
    "BLEU_salient": df_res['BLEU_salient'].mean(),
}

summary_df = pd.DataFrame({
    "Standard": [metrics["METEOR"], metrics["EditDistance"], metrics["BLEU"]],
    "Salient": [metrics["METEOR_salient"], metrics["EditDistance_salient"], metrics["BLEU_salient"]],
}, index=["METEOR", "EditDistance", "BLEU"])

summary_df["Difference"] = summary_df["Standard"] - summary_df["Salient"]
summary_df["% Change"] = ((summary_df["Difference"] / summary_df["Standard"]) * 100).round(2)

summary_df = summary_df.round(4)

summary_df

,Standard,Salient,Difference,% Change
METEOR,0.3710,0.3630,0.0079,2.14
EditDistance,21.6850,21.4745,0.2105,0.97
BLEU,0.1166,0.1110,0.0057,4.88


In [ ]:
correlation_matrix = df_res[['METEOR', 'BLEU', 'EditDistance']].corr()
salient_correlation = df_res[['METEOR', 'METEOR_salient']].corr()

print("\nMetric Correlations:")
print(f"METEOR vs BLEU: {correlation_matrix.loc['METEOR', 'BLEU']:.2f}")
print(f"Edit Distance vs METEOR: {correlation_matrix.loc['EditDistance', 'METEOR']:.2f}")
print(f"Base METEOR vs Salient METEOR: {salient_correlation:.2f}")


Metric Correlations:
METEOR vs BLEU: 0.78
Edit Distance vs METEOR: -0.23


TypeError: unsupported format string passed to DataFrame.__format__

## Approach to Find Polysemous Sentences

### Approach 1
- before adding salient words
- use a python dictionary of polysemous words

In [30]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Download necessary NLTK resources silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)  # Required for Chinese translations in WordNet
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet_ic", quiet=True)

# File paths
en_file_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path = "./zh.translated.desubword"
zh_correct_file_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Common polysemous words (expandable)
polysemous_words = {"bank", "light", "touch", "lead", "bark", "current", "rock", "date", "crane", "seal"}

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        print(f"POS tagging error for word: {word} -> {e}")
        return wn.NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def is_polysemous(word):
    """Check if a word has multiple meanings in WordNet."""
    return len(wn.synsets(word)) > 1

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors():
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            words = set(word_tokenize(en_line.lower()))   # Tokenize sentence
            lemmatized_words = {lemmatize_word(word) for word in words}  # Lemmatize each word
            
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)  # Find polysemous words
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)

                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)

                    if incorrect_mistranslated and not correct_translated:
                        print(f"Possible polysemy-based error detected:")
                        print(f"- English: {en_line}")
                        print(f"- Incorrect Chinese: {zh_line}")
                        print(f"- Correct Chinese: {zh_correct_line}")
                        print(f"- Word causing issue: {word}")
                        print(f"- Possible translations: {possible_translations}")
                        print("-" * 50)

detect_polysemy_errors()


Possible polysemy-based error detected:
- English: And we were particularly touched by the flowers and we were curious as to how the flowers got there.
- Incorrect Chinese: 我们特别受到花朵的触动, 我们很好奇花是怎么摆到那里的。
- Correct Chinese: 特别是那些花,它们尤其让我们感动。 我们很好奇,那些花是怎么摆到那里的呢?”
- Word causing issue: touch
- Possible translations: {'影响', '伸出', '知觉', '涉及', '暗指', '间接提到', '联络', '轻微的侵害', '比得上', '接触', '达到', '少许', '碰到', '身体接触', '摸', '有关', '使接触', '损害', '使相碰', '感觉', '触觉', '碰', '关系到', '触摸', '少量', '作用', '触', '会晤', '触及'}
--------------------------------------------------
Possible polysemy-based error detected:
- English: But on the other hand, we have 14 billion of these: light bulbs, light.
- Incorrect Chinese: 但另一方面, 我们有140亿种灯泡,光亮。
- Correct Chinese: 另一方面, 我们有一百四十亿个 灯泡.
- Word causing issue: light
- Possible translations: {'轻松+地', '容易+地', '点火器', '不足+的', '缺乏+的', '下马', '启发', '明亮+的', '打火机', '昏厥+的', '眩晕+的', '欠缺+的', '毫无约束+的', '放荡+的', '启示', '启蒙', '颜色浅+的', '无意义+的', '点起', '微不足道+的', '点火', '无价值+的', '光源', '光度', '虚弱+的', '轻便+地

### Approach 2
- before adding salient words
- through lemmatization + NLTK's WordNet

In [29]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Ensure NLTK resources are downloaded silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("punkt", quiet=True)

# File paths
en_file_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path = "./zh.translated.desubword"
zh_correct_file_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def get_polysemous_words(min_synsets=2, word_limit=500):
    """
    Extracts polysemous words from WordNet.
    A word is considered polysemous if it has at least `min_synsets` meanings.
    """
    polysemous_words = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().replace('_', ' ')  # Convert WordNet format (e.g., 'rock_n_1' → 'rock')
            if len(wn.synsets(word)) >= min_synsets:  # Ensure multiple meanings
                polysemous_words.add(word)
            if len(polysemous_words) >= word_limit:
                return polysemous_words
    return polysemous_words

# Dynamically generated polysemous words
polysemous_words = get_polysemous_words()

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        return wn.NOUN  # Fallback to NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors():
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            words = set(word_tokenize(en_line.lower()))  # Tokenize and lowercase sentence
            lemmatized_words = {lemmatize_word(word) for word in words}  # Lemmatize words
            
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)  # Find polysemous words
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)

                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)

                    if incorrect_mistranslated and not correct_translated:
                        print(f"Possible polysemy-based error detected:")
                        print(f"- English: {en_line}")
                        print(f"- Incorrect Chinese: {zh_line}")
                        print(f"- Correct Chinese: {zh_correct_line}")
                        print(f"- Word causing issue: {word}")
                        print(f"- Possible translations: {possible_translations}")
                        print("-" * 50)

detect_polysemy_errors()


Possible polysemy-based error detected:
- English: I've been wondering for a long time, since I've been thinking about memes a lot, is there a difference between the memes that we copy -- the words we speak to each other, the gestures we copy, the human things -- and all these technological things around us?
- Incorrect Chinese: 我很久以来在思考迷因是什么, 自从我思考迷因的时候, 我们所复制的迷因之间的差别-- 我们互相说的话, 我们模仿的手势,人类的手势, 以及我们身边所有科技的事物?
- Correct Chinese: 很长一段时间我都在思考, 从我常常思考迷因开始, 我们所复制的迷因之间的差别-- 我们互相说的话, 我们模仿的手势,人类之间的那些事-- 以及所有这些我们周围的技术?
- Word causing issue: long
- Possible translations: {'长+的', '渴望', '有记性+的', '长时间+的', '长久+的', '记性强+的', '冒险的', '能记住+的', '记性好+的', '长期+的', '较长期间+的', '记忆力强+的', '冗长+的', '相对高+的', '久'}
--------------------------------------------------
Possible polysemy-based error detected:
- English: There are very important people, business and land assets in Detroit, and there are real opportunities there.
- Incorrect Chinese: 底特律有很多很重要的人员, 商业和土地资产, 也有真正的机会。
- Correct Chinese: 底特律仍有着十分重要的人群、 产业以及土地, 而

### Comparing before and after adding salient words

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Ensure NLTK resources are downloaded silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("punkt", quiet=True)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def get_polysemous_words(min_synsets=2, word_limit=500):
    """
    Extracts polysemous words from WordNet.
    A word is considered polysemous if it has at least `min_synsets` meanings.
    """
    polysemous_words = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().replace('_', ' ')  # Convert WordNet format (e.g., 'rock_n_1' → 'rock')
            if len(wn.synsets(word)) >= min_synsets:  # Ensure multiple meanings
                polysemous_words.add(word)
            if len(polysemous_words) >= word_limit:
                return polysemous_words
    return polysemous_words

# Dynamically generated polysemous words
polysemous_words = get_polysemous_words()

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        return wn.NOUN  # Fallback to NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors(en_file_path, zh_file_path, zh_correct_file_path):
    errors_detected = []
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line, zh_line, zh_correct_line = en_line.strip(), zh_line.strip(), zh_correct_line.strip()
            words = set(word_tokenize(en_line.lower()))
            lemmatized_words = {lemmatize_word(word) for word in words}
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)
                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)
                    
                    if incorrect_mistranslated and not correct_translated:
                        errors_detected.append({
                            "English Sentence": en_line,
                            "Incorrect Chinese": zh_line,
                            "Correct Chinese": zh_correct_line,
                            "Polysemous Word": word,
                            "Possible Translations": ", ".join(possible_translations)
                        })
    
    return errors_detected

In [ ]:
# File paths before adding salient words
en_file_path_before = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path_before = "./zh.base.translated.desubword"
zh_correct_file_path_before = "./en-zh.zh-filtered.zh.subword.test.desubword"

errors_before = detect_polysemy_errors(en_file_path_before, zh_file_path_before, zh_correct_file_path_before)

pd.DataFrame(errors_before)

,English Sentence,Incorrect Chinese,Correct Chinese,Polysemous Word,Possible Translations
0,"Right now, I am just making my institute in Br...",我们现在对数学教育有个真正的问题。,▁目前我们的数学教育面临着实际的问题。,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
1,I heard my heart's valves snapping open and cl...,"匿名者:美国福克斯新闻,它引起了我们对于 匿名者名字和自然的关注。",匿名:亲爱的福克斯新闻 很不幸得引起了我们的注意▁所有匿名者的名称和性质 已经被破坏,close,"总结, 使靠拢, 亲密+的, 不通风+的, 没有风+的, 接近+地, 曲终人散, 质地细密的..."
2,A world where you're living at the frontier.,"让我给你们一个概括一下, 什么是无人飞行器。",▁让我总结一下▁“禁捕”保护区的益处,living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
3,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
4,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",cut,"稀释, 等级, 不打招呼, 冷落, 凿出的道, 通道, 缩短+的, 剪短, 割, 做, 一份..."
5,"And, in fact, when we did the interview -- I d...","1 加 2 加 3 等于 5, 3 加 5 是 8, 等等.","1 加 2 等于 3 2 加 3 等于 5, 3 加 5 等于 8▁以此类推.",living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
6,"Half the challenge is to get access, is to be ...","我的父亲对法律的尊重受到了极大的尊敬, 尽管他被判刑, 但他从没想过错误论文。",我父亲一直被教导要做守法公民▁虽然他受到迫害▁但从没想过办假证件这回事,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
7,But it's only in the last decade or so that ca...,"但是,你发现的不是人类遗留下的残骸, 比如塞拉姆和露西,每天的。","▁但是你找到的并不是在通常意义上存在的人类, 就像是塞勒姆和露西。",last,"连续, 完结, 延续, 最后阶段, 忍耐, 终点+的, 最後, 过着, 确定性+的, 结束,..."


In [ ]:
# File paths after adding salient words
en_file_path_after = "./en-zh.en-filtered-salient.en.subword.test.desubword"
zh_file_path_after = "./zh.base.translated.desubword"
zh_correct_file_path_after = "./en-zh.zh-filtered.zh.subword.test.desubword"

errors_after = detect_polysemy_errors(en_file_path_after, zh_file_path_after, zh_correct_file_path_after)

pd.DataFrame(errors_after)

,English Sentence,Incorrect Chinese,Correct Chinese,Polysemous Word,Possible Translations
0,"Right now, I am just making my institute in Br...",我们现在对数学教育有个真正的问题。,▁目前我们的数学教育面临着实际的问题。,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
1,I heard my heart's valves snapping open and cl...,"匿名者:美国福克斯新闻,它引起了我们对于 匿名者名字和自然的关注。",匿名:亲爱的福克斯新闻 很不幸得引起了我们的注意▁所有匿名者的名称和性质 已经被破坏,close,"总结, 使靠拢, 亲密+的, 不通风+的, 没有风+的, 接近+地, 曲终人散, 质地细密的..."
2,A world where you're living at the frontier.,"让我给你们一个概括一下, 什么是无人飞行器。",▁让我总结一下▁“禁捕”保护区的益处,living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
3,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
4,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",cut,"稀释, 等级, 不打招呼, 冷落, 凿出的道, 通道, 缩短+的, 剪短, 割, 做, 一份..."
5,"And, in fact, when we did the interview -- I d...","1 加 2 加 3 等于 5, 3 加 5 是 8, 等等.","1 加 2 等于 3 2 加 3 等于 5, 3 加 5 等于 8▁以此类推.",living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
6,"Half the challenge is to get access, is to be ...","我的父亲对法律的尊重受到了极大的尊敬, 尽管他被判刑, 但他从没想过错误论文。",我父亲一直被教导要做守法公民▁虽然他受到迫害▁但从没想过办假证件这回事,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
7,But it's only in the last decade or so that ca...,"但是,你发现的不是人类遗留下的残骸, 比如塞拉姆和露西,每天的。","▁但是你找到的并不是在通常意义上存在的人类, 就像是塞勒姆和露西。",last,"连续, 完结, 延续, 最后阶段, 忍耐, 终点+的, 最後, 过着, 确定性+的, 结束,..."
